# 🧠 Taller: Redes Convolucionales desde Cero (CNN)
## Reconocimiento de Imágenes con Keras (TensorFlow)

**Dataset:** FashionMNIST (imágenes de ropa en escala de grises, 28x28 píxeles)

---

### 📖 ¿Qué vamos a hacer aquí?

Imagina que quieres enseñarle a una computadora a **mirar una foto de ropa y decir qué es** (una camiseta, un zapato, un bolso...). Eso es lo que vamos a construir: un programa que aprende a reconocer imágenes **viendo muchos ejemplos**, igual que un niño aprende a reconocer animales viendo muchos perros y gatos.

La herramienta que usaremos se llama **Red Neuronal Convolucional (CNN)**. No te preocupes por el nombre técnico todavía; lo iremos explicando pieza por pieza.

> 💡 **No necesitas saber nada de matemáticas ni de inteligencia artificial para seguir este notebook.** Cada celda tiene una explicación en palabras simples antes del código.

---

### ⚙️ ANTES DE EMPEZAR — Activa la GPU (¡importante!)

Una GPU es una pieza del computador que hace que el "aprendizaje" sea **mucho más rápido** (minutos en vez de horas). Google Colab te presta una gratis. Para activarla:

1. Ve al menú de arriba: **Entorno de ejecución** → **Cambiar tipo de entorno de ejecución**
2. En "Acelerador por hardware" elige **GPU (T4)**
3. Guarda y listo.

Luego ejecuta las celdas en orden, una por una, con el botón ▶️ (o `Shift + Enter`).

## Paso 0 — Revisar que todo esté listo

Esta celda no hace nada importante todavía: solo **carga las herramientas** que vamos a usar y comprueba si la GPU está activada. Piensa en esto como sacar los ingredientes antes de cocinar.

- `tensorflow` / `keras` → la "cocina" donde se construye y entrena el modelo.
- `numpy` → para manejar números y listas de forma eficiente.
- `matplotlib` → para dibujar imágenes y gráficas.
- `scikit-learn` → para medir qué tan bien funciona el modelo.

In [ ]:
# Cargamos las herramientas (librerías)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print("Versión de TensorFlow:", tf.__version__)

# ¿Tenemos GPU disponible?
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU detectada:", gpus[0].name, "— el entrenamiento será rápido.")
else:
    print("⚠️ No hay GPU. Funcionará igual, pero más lento.")
    print("   Actívala en: Entorno de ejecución → Cambiar tipo de entorno → GPU")

## Paso 1 — Cargar las imágenes

Vamos a usar **FashionMNIST**: una colección de **70.000 fotos pequeñas de ropa** (en blanco y negro), ya etiquetadas por humanos. Cada foto mide 28x28 píxeles (muy pequeña) y pertenece a una de 10 categorías de ropa.

### ¿Qué es "entrenamiento" y "prueba"?

Dividimos las fotos en dos grupos, igual que un profesor con sus alumnos:

- **Entrenamiento (60.000 fotos):** son los "ejercicios de práctica". El modelo las ve muchas veces y aprende de ellas.
- **Prueba (10.000 fotos):** son el "examen final". El modelo **nunca las vio durante el aprendizaje**, así que sirven para medir si de verdad aprendió o solo memorizó.

Esto es clave: si evaluáramos con las mismas fotos del entrenamiento, sería como dar un examen con las mismas preguntas de la práctica — no mediríamos aprendizaje real.

In [ ]:
# Keras ya trae FashionMNIST listo para descargar
fashion_mnist = keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Nombres de las 10 categorías (el dataset usa números 0-9, esto los traduce)
nombres_clases = ['Camiseta/Top', 'Pantalón', 'Pulóver', 'Vestido', 'Abrigo',
                  'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

print("Fotos de entrenamiento:", x_train.shape[0], "| tamaño de cada una:", x_train.shape[1:])
print("Fotos de prueba:       ", x_test.shape[0])
print("Categorías:", len(nombres_clases))

### Veamos algunas imágenes

Antes de entrenar nada, es buena idea **mirar los datos con nuestros propios ojos**. Vamos a dibujar 15 fotos con su etiqueta correcta debajo. Así entendemos qué tipo de imágenes manejamos.

> 📸 **Guarda esta imagen para tu carpeta `media/`** — es una buena captura para el README (muestra cómo se ven los datos).

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_train[i], cmap='gray')   # cmap='gray' = blanco y negro
    plt.title(nombres_clases[y_train[i]], fontsize=9)
    plt.axis('off')
plt.suptitle("Ejemplos del dataset FashionMNIST", fontsize=13)
plt.tight_layout()
plt.savefig("ejemplos_dataset.png", dpi=100, bbox_inches='tight')  # se guarda en el archivo
plt.show()

## Paso 2 — Preparar las imágenes para el modelo

Las computadoras no "ven" imágenes; ven **números**. Cada píxel de una foto en blanco y negro es un número entre **0 (negro)** y **255 (blanco)**. Antes de entrenar, hacemos dos ajustes pequeños pero importantes:

**1. Normalizar (dividir entre 255):** convertimos esos números de 0–255 a un rango de **0 a 1**. ¿Por qué? Porque a las redes neuronales "les cuesta menos" trabajar con números pequeños y parejos — aprenden de forma más estable y rápida. Es como pasar de medir en milímetros gigantes a una escala cómoda.

**2. Añadir un "canal":** las CNN esperan que cada imagen indique cuántos colores tiene. Como son blanco y negro, hay **1 solo canal**. (Una foto a color tendría 3: rojo, verde y azul). Solo le decimos al modelo "esto tiene 1 canal" cambiando la forma de los datos de `(28,28)` a `(28,28,1)`.

In [ ]:
# 1. Normalizar: de 0-255 a 0.0-1.0
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

# 2. Añadir la dimensión del canal (1 = escala de grises)
x_train = np.expand_dims(x_train, -1)   # ahora forma = (60000, 28, 28, 1)
x_test  = np.expand_dims(x_test, -1)

print("Nueva forma de las imágenes de entrenamiento:", x_train.shape)
print("Valor mínimo:", x_train.min(), "| Valor máximo:", x_train.max(), "(antes era 0 a 255)")

## Paso 3 — Construir la red neuronal (la CNN) 🏗️

Esta es la parte central. Vamos a apilar **capas**, una sobre otra, como una torre de bloques. Cada capa hace un trabajo específico. Aquí explico cada bloque con una analogía:

---

### 🔍 `Conv2D` — La capa convolucional (la "lupa que busca patrones")
Imagina pasar una **lupa pequeña** por toda la foto, buscando detalles: bordes, esquinas, líneas, texturas. Eso hace `Conv2D`. Cada lupa se llama **filtro**.
- **filtros = 32:** usamos 32 lupas distintas, cada una busca un patrón diferente.
- **kernel_size = (3,3):** cada lupa "mira" un cuadradito de 3x3 píxeles a la vez.
- **padding = 'same':** hace que la imagen no se encoja en los bordes al pasar la lupa (rellena el borde con ceros).

### ⚡ `ReLU` — La función de activación (el "filtro de lo importante")
Después de cada lupa, ReLU hace algo simple: **deja pasar lo positivo y convierte lo negativo en cero.** Es como decir "si encontraste algo interesante, repórtalo; si no, ignóralo". Esto le da al modelo la capacidad de aprender cosas complejas. (En el código va dentro del `Conv2D` como `activation='relu'`).

### 🔽 `MaxPooling2D` — La capa de agrupamiento (el "resumen")
Reduce el tamaño de la imagen quedándose solo con lo **más fuerte** de cada zona. Si tienes un cuadrito de 2x2, se queda con el valor más grande. ¿Para qué? Para que el modelo sea más rápido y se enfoque en *lo importante*, ignorando detalles diminutos. Es como resumir un párrafo en una frase.

### 📏 `Flatten` — Aplanar
Hasta aquí trabajamos con imágenes en 2D (cuadrículas). `Flatten` las **estira en una sola fila larga** de números, porque la siguiente capa necesita los datos así.

### 🧮 `Dense` — Capa densa (la "que toma la decisión")
Aquí todas las neuronas están conectadas entre sí y combinan toda la información para **decidir qué prenda es**. La última `Dense` tiene exactamente **10 neuronas** (una por cada categoría de ropa).

### 🎯 `softmax` — La última activación (el "que reparte porcentajes")
Convierte la salida final en **probabilidades que suman 100%**. Por ejemplo: "85% camiseta, 10% camisa, 5% otros". El modelo elige la opción con mayor porcentaje.

### 💧 `Dropout` (BONUS) — Evita la "memorización"
Durante el entrenamiento, **apaga al azar** algunas neuronas en cada paso. Suena raro, pero obliga al modelo a no depender de unas pocas neuronas y a aprender de forma más robusta. Esto evita el **overfitting** (cuando el modelo memoriza en vez de aprender). `Dropout(0.25)` apaga el 25% al azar.

In [ ]:
# Construimos el modelo apilando capas en orden (Sequential = "una tras otra")
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),                      # entrada: imagen 28x28 de 1 canal

    # --- Primer bloque: lupa + resumen ---
    layers.Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # --- Segundo bloque: más lupas (64) + resumen ---
    layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # --- Decisión final ---
    layers.Flatten(),                                    # aplanar a una fila
    layers.Dense(128, activation='relu'),                # capa que combina información
    layers.Dropout(0.25),                                # BONUS: evita memorización
    layers.Dense(10, activation='softmax')               # 10 salidas (una por categoría)
])

# Mostramos un resumen de la arquitectura
model.summary()

### 📊 ¿Cómo leer el `summary()` de arriba?

- **Layer (type):** el nombre y tipo de cada capa que apilamos.
- **Output Shape:** la forma de los datos al salir de esa capa. Verás cómo la imagen se va "encogiendo" (28→14→7) por el MaxPooling y luego se aplana.
- **Param #:** cuántos "ajustes internos" (parámetros) tiene cada capa. Estos son los números que el modelo **irá afinando durante el entrenamiento** para aprender. El total suele ser de cientos de miles: ¡eso es lo que se ajusta solo!

## Paso 4 — Entrenar el modelo 🏋️

"Entrenar" significa: mostrarle las 60.000 fotos de práctica **varias veces** y dejar que ajuste sus parámetros internos hasta que acierte cada vez más. Antes de empezar, configuramos tres cosas con `compile()`:

### 🧭 Optimizador: `Adam`
Es la "estrategia de aprendizaje". Decide **cómo y cuánto** ajustar el modelo en cada paso para mejorar. `Adam` es el más popular para empezar porque es estable y casi no necesita configuración. (Existe otro llamado `SGD`, más simple pero más lento de afinar).

### 📉 Función de pérdida: `sparse_categorical_crossentropy`
La "pérdida" (loss) es un número que mide **qué tan equivocado** está el modelo. Cuanto más bajo, mejor. El objetivo del entrenamiento es **bajar este número**. Usamos esta versión específica porque nuestras etiquetas son números enteros (0–9) y tenemos varias categorías.

### ✅ Métrica: `accuracy` (precisión)
Es el porcentaje de aciertos, fácil de entender: "de cada 100 fotos, ¿cuántas adivinó bien?".

### 🔁 ¿Qué es una "época" (epoch)?
Una época = el modelo vio **todas** las fotos de práctica una vez. Haremos **10 épocas**, así que las verá 10 veces, mejorando un poco cada vuelta.

In [ ]:
# Configuramos el entrenamiento
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ¡A entrenar! Esto puede tardar 1-3 minutos con GPU.
# validation_split=0.1 -> aparta el 10% de las fotos de práctica para vigilar el progreso
historia = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,          # procesa de a 128 fotos a la vez (más eficiente)
    validation_split=0.1,
    verbose=1
)
print("\n✅ ¡Entrenamiento terminado!")

## Paso 5 — ¿Cómo fue aprendiendo? (Curvas de entrenamiento)

Durante el entrenamiento, guardamos cómo evolucionaron la **precisión** y la **pérdida** en cada época. Vamos a graficarlas. Esto cuenta una historia:

- La **precisión (accuracy)** debería **subir** con cada época. 📈
- La **pérdida (loss)** debería **bajar** con cada época. 📉
- Comparamos dos líneas: la de **entrenamiento** (las fotos de práctica) y la de **validación** (fotos que el modelo no usa para aprender, solo para vigilarse).

> 🔎 **Señal de alerta (overfitting):** si la línea de entrenamiento sigue mejorando pero la de validación se estanca o empeora, el modelo está *memorizando* en vez de aprender. El `Dropout` que añadimos ayuda a evitar esto.

> 📸 **Guarda esta gráfica para `media/`** — es una de las capturas obligatorias del README.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Gráfica de precisión (accuracy)
ax1.plot(historia.history['accuracy'], 'o-', label='Entrenamiento')
ax1.plot(historia.history['val_accuracy'], 's-', label='Validación')
ax1.set_title('Precisión por época (más alto = mejor)')
ax1.set_xlabel('Época'); ax1.set_ylabel('Precisión')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Gráfica de pérdida (loss)
ax2.plot(historia.history['loss'], 'o-', label='Entrenamiento')
ax2.plot(historia.history['val_loss'], 's-', label='Validación')
ax2.set_title('Pérdida por época (más bajo = mejor)')
ax2.set_xlabel('Época'); ax2.set_ylabel('Pérdida')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("curvas_entrenamiento.png", dpi=100, bbox_inches='tight')
plt.show()

## Paso 6 — El examen final (evaluar con fotos nuevas)

Ahora viene el momento de la verdad: probamos el modelo con las **10.000 fotos de prueba** que **nunca vio**. Este número es el que de verdad importa, porque mide si el modelo sabe reconocer ropa *en general*, no solo las fotos que practicó.

Para FashionMNIST, un buen resultado está alrededor del **90-92%** de aciertos.

In [ ]:
loss_test, acc_test = model.evaluate(x_test, y_test, verbose=0)
print(f"🎯 Precisión en el examen final (test): {acc_test*100:.2f}%")
print(f"   Pérdida en test: {loss_test:.4f}")

## Paso 7 — Matriz de confusión (¿en qué se equivoca?)

Una **matriz de confusión** es una tabla que muestra, para cada categoría real, qué predijo el modelo. Se lee así:

- **La diagonal** (de esquina a esquina) son los **aciertos**: el modelo dijo lo correcto. Queremos que estos números sean grandes.
- **Fuera de la diagonal** están los **errores**: confusiones. Por ejemplo, es típico que confunda "Camisa" con "Camiseta" o "Pulóver" — ¡hasta a nosotros nos costaría diferenciarlas en fotos diminutas en blanco y negro!

Esto es muy útil para tu README: te deja **explicar *por qué* se equivoca**, no solo cuánto.

> 📸 **Guarda esta gráfica para `media/`.**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Pedimos al modelo que prediga las 10.000 fotos de prueba
predicciones = model.predict(x_test, verbose=0)
y_pred = np.argmax(predicciones, axis=1)   # elige la categoría con mayor probabilidad

# Construimos y dibujamos la matriz
matriz = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(10, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=nombres_clases)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title("Matriz de confusión (diagonal = aciertos)")
plt.tight_layout()
plt.savefig("matriz_confusion.png", dpi=100, bbox_inches='tight')
plt.show()

## Paso 8 — Ver aciertos y errores con nuestros ojos

Los números están bien, pero ver ejemplos reales ayuda a entender. Vamos a mostrar:
- Algunas **predicciones correctas** (en verde) ✅
- Algunas **predicciones incorrectas** (en rojo) ❌, donde verás qué confundió con qué.

Debajo de cada foto aparece: lo que el modelo **adivinó** y, entre paréntesis, lo **correcto**.

> 📸 **Otra buena captura para `media/`.**

In [ ]:
# Separamos los índices de aciertos y errores
aciertos   = np.where(y_pred == y_test)[0]
errores    = np.where(y_pred != y_test)[0]

plt.figure(figsize=(13, 6))

# Fila 1: 5 aciertos
for i, idx in enumerate(aciertos[:5]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx].squeeze(), cmap='gray')
    plt.title(f"✅ {nombres_clases[y_pred[idx]]}", color='green', fontsize=9)
    plt.axis('off')

# Fila 2: 5 errores
for i, idx in enumerate(errores[:5]):
    plt.subplot(2, 5, i + 6)
    plt.imshow(x_test[idx].squeeze(), cmap='gray')
    plt.title(f"❌ {nombres_clases[y_pred[idx]]}\n(real: {nombres_clases[y_test[idx]]})",
              color='red', fontsize=8)
    plt.axis('off')

plt.suptitle("Arriba: aciertos | Abajo: errores", fontsize=12)
plt.tight_layout()
plt.savefig("predicciones.png", dpi=100, bbox_inches='tight')
plt.show()

## Paso 9 (BONUS) — Guardar el modelo y volver a cargarlo

Entrenar toma tiempo, así que es útil **guardar** el modelo ya entrenado en un archivo. Luego puedes **cargarlo** en otro momento sin volver a entrenar — listo para usar.

Guardamos en formato `.keras` (el formato moderno de Keras).

In [ ]:
# Guardar el modelo entrenado en un archivo
model.save("modelo_cnn_fashion.keras")
print("💾 Modelo guardado como 'modelo_cnn_fashion.keras'")

# Volver a cargarlo (simulamos que es una sesión nueva)
modelo_cargado = keras.models.load_model("modelo_cnn_fashion.keras")
print("📂 Modelo cargado de nuevo.")

# Comprobamos que funciona igual
_, acc_recargado = modelo_cargado.evaluate(x_test, y_test, verbose=0)
print(f"✅ Precisión del modelo recargado: {acc_recargado*100:.2f}% (debe coincidir con antes)")

## Paso 10 (BONUS) — Experimentar: ¿qué pasa si cambio cosas?

Se pide probar el impacto de cambiar el **tamaño del kernel** o el **número de filtros**. Aquí construimos un segundo modelo más pequeño (kernels de 5x5 y menos filtros) y lo comparamos. No hay respuesta "correcta": el objetivo es **observar y comentar** el cambio en tu README.

> 💡 Idea para comentar: más filtros suelen dar más precisión pero el modelo es más lento; un kernel más grande "mira" zonas más amplias de la imagen.

In [ ]:
# Modelo alternativo: kernel 5x5 y menos filtros (16 y 32)
modelo_b = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (5, 5), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (5, 5), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
modelo_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Lo entrenamos pocas épocas solo para comparar (más rápido)
modelo_b.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1)

_, acc_b = modelo_b.evaluate(x_test, y_test, verbose=0)
print(f"\n📊 COMPARACIÓN:")
print(f"   Modelo original (kernel 3x3, 32-64 filtros): {acc_test*100:.2f}%")
print(f"   Modelo alternativo (kernel 5x5, 16-32 filtros): {acc_b*100:.2f}%")

## 🎉 ¡Terminaste!

### Resumen de lo que construiste
1. Cargaste 70.000 imágenes de ropa y las exploraste.
2. Las preparaste (normalización + canal).
3. Construiste una **CNN** entendiendo cada capa: convolución, ReLU, pooling, flatten, dense, dropout, softmax.
4. La entrenaste y viste sus curvas de aprendizaje.
5. La evaluaste con fotos nuevas (~90% de aciertos).
6. Analizaste sus errores con la matriz de confusión y ejemplos visuales.
7. Guardaste y recargaste el modelo.
8. Experimentaste cambiando la arquitectura.

### 📁 Archivos generados (¡bájalos para tu carpeta `media/`!)
En el panel izquierdo de Colab (ícono de carpeta 📁) encontrarás estas imágenes para descargar:
- `ejemplos_dataset.png`
- `curvas_entrenamiento.png`
- `matriz_confusion.png`
- `predicciones.png`
- `modelo_cnn_fashion.keras` (el modelo, va en `src/`)

Para descargar: clic derecho sobre el archivo → **Descargar**.

### ✍️ Para tu README
- Las 4 imágenes cubren el mínimo de "2 capturas por implementación".
- En **Aprendizajes y dificultades** puedes hablar de: qué es overfitting y cómo lo viste en las curvas, por qué el modelo confunde camisas con camisetas, y qué pasó al cambiar el kernel.

¡Buen trabajo! 🚀